# 🏷️ Label Formulation Analysis: name_only vs description

Compare label formulation impact across 3 datasets × 7 models = 42 experiments

**Datasets**: AG News (4 classes), Banking77 (77 classes), GoEmotions (28 classes)

**Models**: MPNet, BGE-M3, E5-large, Qwen3, Jina v3, INSTRUCTOR, Nomic-MoE

**Note**: Config files are already in `experiments/label_formulation/` directory

## Setup: Clone from GitHub

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone from GitHub
!git clone https://github.com/EngindalgaMaku/zeroshot-text-classification-benchmark-.git
%cd zeroshot-text-classification-benchmark-

# 3. Create symlink for results
import os
drive_results = '/content/drive/MyDrive/zeroshot_results'
os.makedirs(drive_results, exist_ok=True)

if os.path.exists('results'):
    !rm -rf results
!ln -s {drive_results} results
print(f"✅ Results will be saved to: {drive_results}")

# 4. Install packages
!pip install -q -r requirements.txt
print("✅ Setup complete!")

In [ ]:
import torch

print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)} - {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Check Available Experiments

List all label formulation config files

In [ ]:
import glob

# Check what configs we have
configs = sorted(glob.glob("experiments/label_formulation/*.yaml"))
print(f"Found {len(configs)} label formulation config files:\n")

for config in configs:
    exp_name = config.split("/")[-1].replace(".yaml", "").replace("exp_", "")
    print(f"  - {exp_name}")

print(f"\n✅ Total: {len(configs)} experiments")

## Run All Label Formulation Experiments

Run experiments comparing `name_only` vs `description` label modes

In [ ]:
# Settings
SKIP_EXISTING = True

if SKIP_EXISTING:
    print("Mode: --skip-existing (will skip if results exist)")
    print("💡 To re-run all experiments, set SKIP_EXISTING = False\n")
else:
    print("Mode: OVERWRITE (will re-run all experiments)\n")

results = []
for i, config in enumerate(configs, 1):
    exp_name = config.split("/")[-1].replace(".yaml", "")
    print(f"\n{'='*70}")
    print(f"[{i}/{len(configs)}] {exp_name}")
    print(f"{'='*70}")
    
    try:
        cmd = f"python main.py --config {config}"
        if SKIP_EXISTING:
            cmd += " --skip-existing"
        !{cmd}
        results.append((exp_name, "✅"))
    except Exception as e:
        print(f"ERROR: {e}")
        results.append((exp_name, "❌"))

print("\n" + "="*70)
print("EXPERIMENT SUMMARY")
print("="*70)
success = sum(1 for _, s in results if s == "✅")
print(f"Success: {success}/{len(results)}\n")
for exp, status in results:
    print(f"{status} {exp}")

## Load and Organize Results

In [ ]:
import json
import pandas as pd
from pathlib import Path

files = list(Path("results/raw").glob("*_metrics.json"))
data = []

for f in files:
    # Only process label formulation experiments
    if "name_only" not in f.name and not any(ds in f.name for ds in ["agnews", "banking77", "goemotions"]):
        continue
    
    with open(f) as fp:
        m = json.load(fp)
    
    exp_name = m.get("experiment_name", f.stem)
    
    # Determine dataset
    if "agnews" in exp_name or "ag_news" in exp_name:
        dataset = "AG News"
    elif "banking" in exp_name:
        dataset = "Banking77"
    elif "goemotions" in exp_name or "go_emotions" in exp_name:
        dataset = "GoEmotions"
    else:
        continue
    
    # Determine model
    if "mpnet" in exp_name:
        model = "MPNet"
    elif "jina" in exp_name:
        model = "Jina v3"
    elif "qwen3" in exp_name:
        model = "Qwen3"
    elif "bge" in exp_name:
        model = "BGE-M3"
    elif "e5" in exp_name:
        model = "E5-large"
    elif "instructor" in exp_name:
        model = "INSTRUCTOR"
    elif "nomic" in exp_name:
        model = "Nomic-MoE"
    else:
        model = "Unknown"
    
    # Determine label mode
    label_mode = "name_only" if "name_only" in exp_name else "description"
    
    data.append({
        "dataset": dataset,
        "model": model,
        "label_mode": label_mode,
        "macro_f1": m["macro_f1"] * 100,
        "accuracy": m["accuracy"] * 100,
        "samples": m["num_samples"],
    })

df = pd.DataFrame(data)
print(f"\nLoaded {len(df)} results")
print(df.head(10))

## Analysis: Label Mode Comparison

In [ ]:
# Create comparison table
comparison = df.pivot_table(
    index=["dataset", "model"],
    columns="label_mode",
    values="macro_f1"
).reset_index()

# Calculate difference (description - name_only)
comparison["difference"] = comparison["description"] - comparison["name_only"]
comparison["improvement_%"] = (comparison["difference"] / comparison["name_only"]) * 100

print("\n" + "="*70)
print("LABEL FORMULATION COMPARISON (Macro F1 %)")
print("="*70)
print(comparison.to_string(index=False))

# Save to CSV
comparison.to_csv("results/label_formulation_comparison.csv", index=False)
print("\n✅ results/label_formulation_comparison.csv")

In [ ]:
# Summary statistics by dataset
print("\n" + "="*70)
print("AVERAGE IMPROVEMENT BY DATASET")
print("="*70)
dataset_summary = comparison.groupby("dataset")[["difference", "improvement_%"]].mean()
print(dataset_summary.to_string())

print("\n" + "="*70)
print("AVERAGE IMPROVEMENT BY MODEL")
print("="*70)
model_summary = comparison.groupby("model")[["difference", "improvement_%"]].mean()
print(model_summary.sort_values("difference", ascending=False).to_string())

## Visualization: Bar Chart Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# Create subplots for each dataset
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
datasets = ["AG News", "Banking77", "GoEmotions"]

for idx, dataset in enumerate(datasets):
    ax = axes[idx]
    dataset_data = comparison[comparison["dataset"] == dataset]
    
    x = np.arange(len(dataset_data))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, dataset_data["name_only"], width, label="name_only", color="#FF6B6B")
    bars2 = ax.bar(x + width/2, dataset_data["description"], width, label="description", color="#4ECDC4")
    
    ax.set_xlabel("Model", fontsize=12)
    ax.set_ylabel("Macro F1 (%)", fontsize=12)
    ax.set_title(f"{dataset}", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(dataset_data["model"], rotation=45, ha="right")
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 100)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle("Label Formulation Impact: name_only vs description", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("results/plots/label_formulation_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ results/plots/label_formulation_comparison.png")

## Visualization: Improvement Heatmap

In [ ]:
# Create heatmap of improvements
fig, ax = plt.subplots(figsize=(10, 8))

improvement_pivot = comparison.pivot(index="model", columns="dataset", values="difference")

sns.heatmap(improvement_pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            cbar_kws={'label': 'F1 Improvement (description - name_only)'},
            linewidths=0.5, ax=ax)

ax.set_title("Label Formulation Impact by Model and Dataset", fontsize=14, fontweight="bold", pad=20)
ax.set_xlabel("Dataset", fontsize=12)
ax.set_ylabel("Model", fontsize=12)

plt.tight_layout()
plt.savefig("results/plots/label_formulation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ results/plots/label_formulation_heatmap.png")

## Analysis: Which Task Types Benefit Most?

In [ ]:
# Analyze by task characteristics
task_characteristics = {
    "AG News": {"num_classes": 4, "type": "Topic Classification", "complexity": "Low"},
    "Banking77": {"num_classes": 77, "type": "Intent Classification", "complexity": "Very High"},
    "GoEmotions": {"num_classes": 28, "type": "Emotion Classification", "complexity": "High"}
}

print("\n" + "="*70)
print("TASK TYPE ANALYSIS")
print("="*70)

for dataset in datasets:
    avg_improvement = comparison[comparison["dataset"] == dataset]["difference"].mean()
    task_info = task_characteristics[dataset]
    
    print(f"\n{dataset}:")
    print(f"  Type: {task_info['type']}")
    print(f"  Classes: {task_info['num_classes']}")
    print(f"  Complexity: {task_info['complexity']}")
    print(f"  Avg Improvement: {avg_improvement:.2f} F1 points")
    print(f"  Benefit: {'HIGH' if avg_improvement > 5 else 'MEDIUM' if avg_improvement > 2 else 'LOW'}")

In [ ]:
# Correlation between number of classes and improvement
dataset_improvements = comparison.groupby("dataset").agg({
    "difference": "mean"
}).reset_index()

dataset_improvements["num_classes"] = dataset_improvements["dataset"].map(
    lambda x: task_characteristics[x]["num_classes"]
)

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(dataset_improvements["num_classes"], dataset_improvements["difference"], 
           s=200, alpha=0.6, color="#4ECDC4")

for idx, row in dataset_improvements.iterrows():
    ax.annotate(row["dataset"], (row["num_classes"], row["difference"]),
                xytext=(5, 5), textcoords="offset points", fontsize=11, fontweight="bold")

ax.set_xlabel("Number of Classes", fontsize=12)
ax.set_ylabel("Average F1 Improvement (description - name_only)", fontsize=12)
ax.set_title("Label Description Benefit vs Task Complexity", fontsize=14, fontweight="bold")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/plots/label_formulation_vs_complexity.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ results/plots/label_formulation_vs_complexity.png")

## Key Findings Summary

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS: LABEL FORMULATION ANALYSIS")
print("="*70)

# Overall statistics
overall_avg = comparison["difference"].mean()
overall_std = comparison["difference"].std()
positive_improvements = (comparison["difference"] > 0).sum()
total_comparisons = len(comparison)

print(f"\n1. OVERALL IMPACT:")
print(f"   - Average improvement: {overall_avg:.2f} F1 points")
print(f"   - Standard deviation: {overall_std:.2f}")
print(f"   - Positive improvements: {positive_improvements}/{total_comparisons} ({100*positive_improvements/total_comparisons:.1f}%)")

# Best and worst performers
best_improvement = comparison.loc[comparison["difference"].idxmax()]
worst_improvement = comparison.loc[comparison["difference"].idxmin()]

print(f"\n2. BEST IMPROVEMENT:")
print(f"   - {best_improvement['model']} on {best_improvement['dataset']}")
print(f"   - Improvement: {best_improvement['difference']:.2f} F1 points ({best_improvement['improvement_%']:.1f}%)")

print(f"\n3. WORST IMPROVEMENT:")
print(f"   - {worst_improvement['model']} on {worst_improvement['dataset']}")
print(f"   - Change: {worst_improvement['difference']:.2f} F1 points ({worst_improvement['improvement_%']:.1f}%)")

# Dataset-specific insights
print(f"\n4. DATASET-SPECIFIC INSIGHTS:")
for dataset in datasets:
    dataset_avg = comparison[comparison["dataset"] == dataset]["difference"].mean()
    print(f"   - {dataset}: {dataset_avg:.2f} F1 points average improvement")

# Model-specific insights
print(f"\n5. MODEL-SPECIFIC INSIGHTS:")
model_avg = comparison.groupby("model")["difference"].mean().sort_values(ascending=False)
for model, avg in model_avg.items():
    print(f"   - {model}: {avg:.2f} F1 points average improvement")

# Task complexity correlation
print(f"\n6. TASK COMPLEXITY CORRELATION:")
print(f"   - Low complexity (AG News, 4 classes): {comparison[comparison['dataset']=='AG News']['difference'].mean():.2f} F1")
print(f"   - High complexity (GoEmotions, 28 classes): {comparison[comparison['dataset']=='GoEmotions']['difference'].mean():.2f} F1")
print(f"   - Very high complexity (Banking77, 77 classes): {comparison[comparison['dataset']=='Banking77']['difference'].mean():.2f} F1")
print(f"   - Conclusion: {'More complex tasks benefit more from descriptions' if comparison[comparison['dataset']=='Banking77']['difference'].mean() > comparison[comparison['dataset']=='AG News']['difference'].mean() else 'Simpler tasks benefit more from descriptions'}")

print("\n" + "="*70)

## ✅ Analysis Complete!

### Outputs Generated:
1. **results/label_formulation_comparison.csv** - Detailed comparison table
2. **results/plots/label_formulation_comparison.png** - Bar chart comparison
3. **results/plots/label_formulation_heatmap.png** - Improvement heatmap
4. **results/plots/label_formulation_vs_complexity.png** - Complexity correlation

### Key Questions Answered:
- Does label formulation (name_only vs description) impact performance?
- Which models benefit most from label descriptions?
- Which task types benefit most from descriptions?
- Is there a correlation between task complexity and description benefit?

### For TACL Paper:
- Use comparison table in main results section
- Include bar chart or heatmap as main figure
- Discuss task complexity correlation in analysis section
- Highlight model-specific patterns in discussion